# LifeMate Inference & Timing Notebook (Zalo AI Challenge 2026)

Cấu trúc 4 bước theo Phụ lục Mục 7 của BTC:
1. **Set Seed & Environment** (trước khi load model)
2. **Load ASR & LLM Models** vào RAM/VRAM cùng lúc trong 1 process và đo `load_seconds`
3. **Read Dataset & Import `ToolEnv`** trực tiếp từ `/data/<split>/zaic_tools`
4. **Loop từng sample theo đúng thứ tự gốc**: đo `t1 = time.perf_counter()` trước khi đọc audio, tạo 1 `ToolEnv`, chạy `ASR -> LLM -> ToolEnv`, ghi & flush từng dòng ra `/result/jupyter_submission.jsonl`, và thống kê thời gian

In [ ]:
# Bước 1: Set Seed & Environment (trước khi load bất kỳ model nào)
import json
import os
import sys
import time
from pathlib import Path

CODE_DIR = Path.cwd() if (Path.cwd() / "lifemate").is_dir() else Path("/code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from lifemate.predictor import (
    LifeMatePredictor,
    PredictorConfig,
    _TimedASRProxy,
    compare_submissions,
    configure_offline_env,
    import_split_tool_env,
    resolve_default_models,
    resolve_split_layout,
    seed_everything,
    timing_summary,
)
from lifemate.catalog import load_catalog
from lifemate.transcripts import read_jsonl

configure_offline_env()

SEED = int(os.environ.get("LIFEMATE_SEED", "42"))
seed_everything(SEED)
print(f"[Step 1/4] Seed fixed at {SEED} | CODE_DIR={CODE_DIR}")

In [ ]:
# Bước 2: Load ASR (CPU) và LLM (GPU) cùng lúc trong 1 process (giữ nguyên toàn bộ hyperparameters với predict.py)
asr_model, llm_model, llm_adapter = resolve_default_models(CODE_DIR)
device = os.environ.get("LIFEMATE_DEVICE", "cuda")

cfg = PredictorConfig(
    asr_model=asr_model.resolve(),
    llm_model=llm_model.resolve(),
    llm_adapter=llm_adapter.resolve() if llm_adapter else None,
    device=device,
    asr_precision="auto",
    asr_decoding_method="modified_beam_search",
    asr_beam_size=8,
    asr_hotwords="auto",
    asr_hotwords_score=1.5,
    asr_post_normalize=True,
    asr_threads=8,
    max_calls=8,
    max_invalid=3,
    seed=SEED,
)

predictor = LifeMatePredictor(cfg)
print(
    f"[Step 2/4] Loaded ASR + LLM in {predictor.load_seconds:.2f}s "
    f"(ASR={predictor.asr_load_seconds:.2f}s, LLM={predictor.llm_load_seconds:.2f}s)"
)

In [ ]:
# Bước 3: Đọc file test cases và import ToolEnv từ đúng thư mục split của BTC
ROOT = Path(os.environ.get("LIFEMATE_DATA_ROOT", "/data"))
SPLIT_ARG = os.environ.get("LIFEMATE_SPLIT", "auto")
OUTPUT_PATH = Path(os.environ.get("LIFEMATE_OUTPUT", "/result/jupyter_submission.jsonl"))
LIMIT = int(os.environ.get("LIFEMATE_LIMIT", "0"))

SPLIT, split_dir, env_root, holder = resolve_split_layout(ROOT, SPLIT_ARG)
ToolEnv = import_split_tool_env(split_dir)
catalog = load_catalog(split_dir / f"{SPLIT}_tools_catalog.json")
test_cases = read_jsonl(split_dir / f"{SPLIT}.jsonl")
if LIMIT > 0:
    test_cases = test_cases[:LIMIT]

print(f"[Step 3/4] SPLIT={SPLIT} | Samples={len(test_cases)} | OUTPUT={OUTPUT_PATH}")

In [ ]:
# Bước 4: Loop từng sample (t1 trước khi đọc audio -> ToolEnv -> ASR -> LLM -> flush jsonl)
asr_proxy = _TimedASRProxy(predictor.asr)
agent = predictor.build_agent(catalog, asr_override=asr_proxy)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
times_ms = []
asr_ms_list = []
predict_t0 = time.perf_counter()

try:
    with OUTPUT_PATH.open("w", encoding="utf-8") as fout:
        for idx, test_case in enumerate(test_cases, 1):
            record, _ = predictor.predict_sample(
                test_case,
                ToolEnv,
                split_dir,
                SPLIT,
                env_root,
                agent,
                asr_proxy=asr_proxy,
            )
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            fout.flush()
            times_ms.append(record["time"])
            asr_ms_list.append(asr_proxy.seconds * 1000.0)
            if idx <= 3 or idx % 20 == 0 or idx == len(test_cases):
                print(
                    f"[{idx}/{len(test_cases)}] {record['id']}: "
                    f"{len(record['tool_calls'])} calls, {record['time'] / 1000.0:.2f}s"
                )
finally:
    total_predict_seconds = time.perf_counter() - predict_t0
    if holder is not None:
        holder.cleanup()
    predictor.close()

stats = timing_summary(
    times_ms,
    load_seconds=predictor.load_seconds,
    asr_load_seconds=predictor.asr_load_seconds,
    llm_load_seconds=predictor.llm_load_seconds,
    total_predict_seconds=total_predict_seconds,
    asr_ms_list=asr_ms_list,
)
print(json.dumps(stats, indent=2, ensure_ascii=False))

ref_submission = OUTPUT_PATH.parent / "submission.jsonl"
if ref_submission.is_file():
    parity = compare_submissions(ref_submission, OUTPUT_PATH)
    print(f"[Parity vs {ref_submission.name}] {json.dumps(parity, ensure_ascii=False)}")